In [1]:
import os
import sys

# 设置缓存目录路径
cache_dir = r'F:\Anaconda\envs\pytorch\cache'

# 确保目录存在
os.makedirs(cache_dir, exist_ok=True)

# 设置所有相关环境变量
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir
os.environ['HUGGINGFACE_HUB_CACHE'] = cache_dir

print(f"设置的缓存目录: {cache_dir}")

# 现在导入transformers
from transformers import pipeline
from transformers.utils import TRANSFORMERS_CACHE

print(f"实际缓存路径: {TRANSFORMERS_CACHE}")

# 验证路径是否正确
if cache_dir == TRANSFORMERS_CACHE:
    print("✅ 缓存路径设置成功！")
else:
    print(f"❌ 缓存路径设置失败！期望: {cache_dir}, 实际: {TRANSFORMERS_CACHE}")

# 运行情感分析
classifier = pipeline("sentiment-analysis")
result = classifier(
    [
        "I've been waiting for a HuggingFace course my whole life.",
        "I hate this so much!",
    ]
)

print(f"\n分析结果: {result}")

# 检查缓存目录是否创建了文件
print(f"\n缓存目录内容:")
if os.path.exists(cache_dir):
    items = os.listdir(cache_dir)
    if items:
        for item in items:
            item_path = os.path.join(cache_dir, item)
            if os.path.isdir(item_path):
                print(f"  📁 {item}")
            else:
                print(f"  📄 {item}")
    else:
        print("   (空目录)")
else:
    print("   (目录不存在)")

import os

# 检查并修复缓存目录权限
cache_dir = r"F:\Anaconda\envs\pytorch\cache"
if os.path.exists(cache_dir):
    import subprocess
    # 给当前用户完全控制权限
    subprocess.run(['icacls', cache_dir, '/grant', 'Everyone:F', '/T'], shell=True)

设置的缓存目录: F:\Anaconda\envs\pytorch\cache


f:\Anaconda\envs\pytorch\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.


实际缓存路径: F:\Anaconda\envs\pytorch\cache
✅ 缓存路径设置成功！



Device set to use cuda:0



分析结果: [{'label': 'POSITIVE', 'score': 0.9598046541213989}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

缓存目录内容:
  📁 .locks
  📁 datasets
  📁 datasets--bigcode--the-stack
  📁 datasets--code_search_net
  📁 datasets--glue
  📁 datasets--lewtun--github-issues
  📁 evaluate
  📁 metrics
  📁 models--bert-base-cased
  📁 models--bert-base-uncased
  📁 models--camembert-base
  📁 models--dbmdz--bert-large-cased-finetuned-conll03-english
  📁 models--distilbert--distilbert-base-cased-distilled-squad
  📁 models--distilbert--distilbert-base-uncased-finetuned-sst-2-english
  📁 models--distilbert-base-cased-distilled-squad
  📁 models--gpt2
  📁 models--roberta-base
  📁 models--sentence-transformers--multi-qa-mpnet-base-dot-v1
  📁 models--WenjieSHI--dummy-model
  📁 modules
  📁 xet


标准化和预分词

在深入探讨 Transformer 模型常用的三种分词算法（字节对编码[BPE]、WordPiece 和 Unigram）之前，我们首先来看看 tokenizer 对文本进行了哪些预处理。以下是 tokenization 过程的高度概括：

在分词（根据其模型）之前，tokenizer 需要进行两个步骤： 标准化（normalization） 和 预分词（pre-tokenization） 。

标准化（normalization）

标准化步骤涉及一些常规清理，例如删除不必要的空格、小写和“/”或删除重音符号。如果你熟悉 Unicode 标准化 （例如 NFC 或 NFKC），那么这也是 tokenizer 可能会使用的东西。

🤗 Transformers 的 tokenizer 具有一个名为 backend_tokenizer 的属性，该属性可以访问来自🤗 Tokenizers 库的底层 tokenizer 。

In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
print(type(tokenizer.backend_tokenizer))

<class 'tokenizers.Tokenizer'>


tokenizer 对象的 normalizer 属性具有一个 normalize_str() 方法，我们可以使用该方法查看如何进行标准化：

In [3]:
print(tokenizer.backend_tokenizer.normalizer.normalize_str("Héllò hôw are ü?"))

hello how are u?


在这个例子中，由于我们选择了 bert-base-uncased checkpoint，所以会在标准化的过程中转换为小写并删除重音。

✏️ 试试看！ 从 bert-base-cased checkpoint 加载 tokenizer 并处理相同的示例。看一看 tokenizer 的 cased 和 uncased 版本之间的主要区别是什么？

In [4]:
from transformers import AutoTokenizer

# 加载两种tokenizer
cased_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
uncased_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

example = "Please do it carefully, otherwise we will never work with Hugging Face again!!"

# 分词处理
cased_tokens = cased_tokenizer.tokenize(example)
uncased_tokens = uncased_tokenizer.tokenize(example)

# 打印对比
print("bert-base-cased tokens:")
print(cased_tokens)
print(f"Token count: {len(cased_tokens)}")

print("\nbert-base-uncased tokens:")
print(uncased_tokens)
print(f"Token count: {len(uncased_tokens)}")

print("\n关键区别:")
print(f"1. 'Hugging' 分词:")
print(f"   cased: {[t for t in cased_tokens if 'Hu' in t or 'gging' in t]}")
print(f"   uncased: {[t for t in uncased_tokens if 'hu' in t or 'gging' in t]}")

print(f"\n2. 'Face' 分词:")
print(f"   cased: 'Face'")
print(f"   uncased: 'face'")

print(f"\n3. 特殊标记格式:")
print(f"   cased: [CLS], [SEP]")
print(f"   uncased: [CLS], [SEP] (相同)")

bert-base-cased tokens:
['Please', 'do', 'it', 'carefully', ',', 'otherwise', 'we', 'will', 'never', 'work', 'with', 'Hu', '##gging', 'Face', 'again', '!', '!']
Token count: 17

bert-base-uncased tokens:
['please', 'do', 'it', 'carefully', ',', 'otherwise', 'we', 'will', 'never', 'work', 'with', 'hugging', 'face', 'again', '!', '!']
Token count: 16

关键区别:
1. 'Hugging' 分词:
   cased: ['Hu', '##gging']
   uncased: ['hugging']

2. 'Face' 分词:
   cased: 'Face'
   uncased: 'face'

3. 特殊标记格式:
   cased: [CLS], [SEP]
   uncased: [CLS], [SEP] (相同)


预分词

正如我们将在下一节中看到的，tokenizer 一般不会在原始文本上进行训练。因此，我们首先需要将文本拆分为更小的实体，例如单词。这就是预分词步骤的作用。正如我们在 第二章 中看到的，基于单词的 tokenizer 可以简单地根据空格和标点符号将原始文本拆分为单词。这些词将是 tokenizer 在训练期间可以学习的子词的边界。

要查看快速 tokenizer 如何执行预分词，我们可以使用 tokenizer 对象的 pre_tokenizer 属性的 pre_tokenize_str() 方法：

In [5]:
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")

[('Hello', (0, 5)),
 (',', (5, 6)),
 ('how', (7, 10)),
 ('are', (11, 14)),
 ('you', (16, 19)),
 ('?', (19, 20))]

请注意 tokenizer 记录了偏移量，这是就是我们在前一节中使用的偏移映射。在这里 tokenizer 将两个空格并将它们替换为一个，从 are 和 you 之间的偏移量跳跃可以看出来这一点。

由于我们使用的是 BERT tokenizer 所以预分词会涉及到在空白和标点上进行分割。其他的 tokenizer 可能会对这一步有不同的规则。例如，如果我们使用 GPT-2 的 tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")

[('Hello', (0, 5)),
 (',', (5, 6)),
 ('Ġhow', (6, 10)),
 ('Ġare', (10, 14)),
 ('Ġ', (14, 15)),
 ('Ġyou', (15, 19)),
 ('?', (19, 20))]

它也会在空格和标点符号上拆分，但它会保留空格并将它们替换为 Ġ 符号，如果我们对 tokens 进行解码，则使其能够恢复原始空格：

也请注意，与 BERT tokenizer 不同的是，这个 tokenizer 不会忽略双空格。

最后一个例子，让我们看一下基于 SentencePiece 算法的 T5 tokenizer

In [7]:
tokenizer = AutoTokenizer.from_pretrained("t5-small")
tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str("Hello, how are  you?")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[('▁Hello,', (0, 6)),
 ('▁how', (7, 10)),
 ('▁are', (11, 14)),
 ('▁you?', (16, 20))]

与 GPT-2 的 tokenizer 类似，这个 tokenizer 保留空格并用特定 token 替换它们（ _ ），但 T5 tokenizer 只在空格上拆分，不考虑标点符号。另外，它会在句子开头（在 Hello 之前）默认添加一个空格，并忽略 are 和 you 之间的双空格。

现在我们对一些不同的 tokenizer 如何处理文本有了一些了解，接下来我们就可以开始探索底层算法本身。我们将先简要了解一下广泛适用的 SentencePiece；然后，在接下来的三节中，我们将研究用于子词分词的三种主要算法的工作原理。

SentencePiece
SentencePiece 是一种用于文本预处理的 tokenization 算法，你可以将其与我们将在接下来的三个部分中看到的任何模型一起使用。它将文本视为 Unicode 字符序列，并用特殊字符 ▁ 替换空格。与 Unigram 算法结合使用（参见 第七节 ）时，它甚至不需要预分词步骤，这对于不使用空格字符的语言（如中文或日语）非常有用。

SentencePiece 的另一个主要特点是可逆 tokenization：由于没有对空格进行特殊处理，解码 tokens 的时候只需将它们连接起来，然后将 _ 替换为空格，就可以还原原始的文本。如我们之前所见，BERT tokenizer 会删除重复的空格，所以它的分词不是可逆的。

算法概述


在下面的部分中，我们将深入研究三种主要的子词 tokenization 算法：BPE（由 GPT-2 等使用）、WordPiece（由 BERT 使用）和 Unigram（由 T5 等使用）。在我们开始之前，先来快速了解它们各自的工作方式。如果你还不能理解，不妨在阅读接下来的每一节之后返回此表格进行查看。

模型	BPE	        WordPiece	    Unigram
训练	从小型词汇表开始，学习合并 token 的规则	从小型词汇表开始，学习合并 token 的规则	从大型词汇表开始，学习删除 token 的规则
训练步骤	合并对应最常见的 token 对	合并对应得分最高的 token 对，优先考虑每个独立 token 出现频率较低的对	删除会在整个语料库上最小化损失的词汇表中的所有 token
学习	合并规则和词汇表	仅词汇表	含有每个 token 分数的词汇表
编码	将一个单词分割成字符并使用在训练过程中学到的合并	从开始处找到词汇表中的最长子词，然后对其余部分做同样的事	使用在训练中学到找到最可能的 token 分割方式